# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library. We focus on using Croissant `@id` identifiers to reference all data entities, ensuring reproducibility and clarity.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading

We load the metadata and records from the dataset using `mlcroissant`. This step fetches both the dataset description and access to its record sets for programmatic analysis.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Get name and description from the metadata object
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

Let's review the available record sets, their fields, and the relevant `@id` values. This helps decide what data to extract for downstream analysis. Each record set and field is referenced by its Croissant `@id`.

In [ ]:
# List available record sets and their fields/columns with their @id.
record_set_summaries = {}
print("Available record sets and their fields:")
for record_set in dataset.metadata.record_sets:
    print(f"\nRecord Set Name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print("  Fields (columns):")
    record_set_summaries[record_set.id] = {'name': record_set.name, 'fields': []}
    for field in record_set.fields:
        # Some fields may not have 'columns', just use id and name
        print(f"    - {field.name} (@id: {field.id})")
        record_set_summaries[record_set.id]['fields'].append({'name': field.name, 'id': field.id})
    print(f"  Example records from this set:")
    example_records = []
    # Show up to 2 records as example, if available
    try:
        for i, rec in enumerate(dataset.records(record_set=record_set.id)):
            if i >= 2:
                break
            print(f"    {json.dumps(rec)[:300]}...\n")
    except Exception as e:
        print(f"    (Error when displaying records: {e})")

## 3. Data Extraction

We'll load all available record sets into Pandas DataFrames for analysis, referencing the sets by their Croissant `@id`. Choose the appropriate record set and field IDs from the output above for further analysis.

In [ ]:
# Extract all record set IDs
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id} with shape {df.shape}")
        print(f"Columns: {list(df.columns)}\n")
    else:
        print(f"No records found for record set id: {record_set_id}\n")
# Display top 5 rows of the first (main) data set for illustration
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's choose example numeric and group fields by their Croissant `@id` from above, and perform basic filtering, normalization, and aggregation.

In [ ]:
# ---- You may adjust these variables based on the actual fields/columns names & @ids from Section 2 ----
selected_record_set_id = main_record_set_id  # For illustration, use the main record set
df = dataframes[selected_record_set_id]

# Guess likely numeric and group fields based on typical mass spec dataset column names
# You can adjust the next two lines based on the real fields
numeric_field_candidates = [c for c in df.columns if c.lower() in [
    'coverage (%)', 'coverage', 'peptide count', 'mw', 'molecular weight', 'abundance', 'normalized abundance', 'pI', 'peptides', 'mw (kda)', 'unique peptides'
]]
numeric_field = numeric_field_candidates[0] if numeric_field_candidates else df.select_dtypes('number').columns[0]

group_field_candidates = [
    c for c in df.columns if c.lower() in ['accession', 'protein', 'sample', 'sample id', 'condition', 'group']
]
group_field = group_field_candidates[0] if group_field_candidates else df.columns[0]

print(f"Using numeric field: {numeric_field} (@id: refer in Section 2)")
print(f"Using group field: {group_field} (@id: refer in Section 2)\n")

# Filter records where the numeric field > threshold
threshold = 10
filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold} (showing 5 rows):")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (pd.to_numeric(filtered_df[numeric_field], errors='coerce') - pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()
print(f"Normalized {numeric_field} (showing first 5 rows):")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by group_field, show aggregate
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped by {group_field} (showing up to 5):")
    display(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the selected numeric field, and relationships between variables, using the extracted DataFrame.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
pd.to_numeric(df[numeric_field], errors='coerce').hist(bins=30)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot grouped by the group field (if few unique values)
if df[group_field].nunique() < 40:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field, y=pd.to_numeric(df[numeric_field], errors='coerce'), data=df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=60)
    plt.show()

# Scatter plot between two numeric fields, if available
numeric_candidates = df.select_dtypes(include='number').columns.drop(numeric_field, errors='ignore')
if len(numeric_candidates) > 0:
    plt.figure(figsize=(6, 4))
    plt.scatter(pd.to_numeric(df[numeric_field], errors='coerce'), pd.to_numeric(df[numeric_candidates[0]], errors='coerce'), alpha=0.6)
    plt.xlabel(numeric_field)
    plt.ylabel(numeric_candidates[0])
    plt.title(f'Scatter: {numeric_field} vs {numeric_candidates[0]}')
    plt.show()

## 6. Conclusion

In this notebook, we've demonstrated systematic programmatic exploration of the FAIR<sup>2</sup> mass spectrometry dataset, using `mlcroissant`, with all references grounded by Croissant `@id` fields. By extracting and analyzing one or more of the record sets, we've previewed the data model, filtered and normalized core numeric fields such as protein abundance/coverage, and produced basic visualizations for rapid scientific evidence assessment.

Further steps could include more sophisticated feature engineering, integrating with public proteomics references, or building statistical/protein classifiers. Please refer to the dataset schema at the Croissant URL for full details on field definitions and citations.